# Pandas Pivot Tables Lab - Restaurant Tipping Data

- For reference, you can use the pivot_table slides on Canvas. Docs links are also attached. 
- Submit this notebook before the end of class.

In this lab you will use **`pd.pivot_table()`** to answer real business questions about restaurant tipping behavior. The dataset was collected by a food server over the course of several months and records every table they served.

---

In [ ]:
# importing libraries and loading datset
# run this cell first

import pandas as pd
import numpy as np
import seaborn as sns

df = sns.load_dataset('tips')
df.head()

**Dataset columns:**

| Column | Description |
|---|---|
| `total_bill` | Total bill amount in dollars |
| `tip` | Tip amount in dollars |
| `sex` | Sex of the bill payer (`Male` / `Female`) |
| `smoker` | Whether the party sat in the smoking section (`Yes` / `No`) |
| `day` | Day of the week (`Thur`, `Fri`, `Sat`, `Sun`) |
| `time` | Meal period (`Lunch` / `Dinner`) |
| `size` | Number of people in the party |

---

## Problem 1: Tipping by Day and Smoking Section

The manager wants to know whether customers in the smoking section tip differently depending on the day.

Create a pivot table showing the **mean tip amount** for each **day** (rows) broken out by **smoker** status (columns).

Store the result in a variable called `pivot1`.

**Expected shape:** (4 rows × 2 columns)

**Boilerplate code:**
```python
pivot1 = pd.pivot_table(df, values=___, index=____, columns=___, aggfunc=_____)

"""
values is the name of the column in df that you are aggregating
index is the name of the column in df that you want your rows to be represented by
columns is the categorical column in df that you want to fan out in the pivot table
aggfunc is the aggregation method ('mean', 'count', 'max', etc.)
"""
```

See pivot_table slides on Canvas for reference.

In [ ]:
# YOUR CODE HERE
pivot1 = pd.pivot_table(
    df,
    values='tip',
    index='day',
    columns='smoker',
    aggfunc='mean'
)
    
pivot1

In [ ]:
# Tests for Problem 1
assert isinstance(pivot1, pd.DataFrame), "pivot1 should be a DataFrame"
assert pivot1.shape == (4, 2), f"Expected shape (4, 2), got {pivot1.shape}"
assert pivot1.columns.name == 'smoker', "Columns should be labeled by 'smoker'"
assert set(pivot1.columns.tolist()) == {'Yes', 'No'}, "Columns should be 'Yes' and 'No'"
assert pivot1.index.name == 'day', "Index should be named 'day'"
assert (pivot1.values > 0).all(), "All mean tip values should be positive"
print("✓ All tests passed for Problem 1!")

---
## Problem 2: Revenue Breakdown with Grand Totals

Accounting needs a revenue summary. Create a pivot table showing **total bill revenue** by **time of day** (rows: `Lunch` / `Dinner`) and **day** (columns), **including row and column grand totals**.

- Put `margins=True` in the `pivot_table` function, 
and label the totals row/column 'Total' by putting `margins_name = 'Total'` in the `pivot_table` function.
- Store the result in `pivot2`

[Example](https://softhints.com/how-to-fix-pandas-pivot_table-margins-aggregating-both-rows-and-columns/)

**Boilerplate code:**
`pivot2 = pd.pivot_table(df, values='___', index='___', columns='___', aggfunc='___', margins=True, margins_name='___'
)`

**Expected shape:** (3 rows × 5 columns) — 2 meal periods + Total × 4 days + Total

In [ ]:
# YOUR CODE HERE
pivot2 = pd.pivot_table(
    df,
    values='total_bill',
    index='time',
    columns='day',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

pivot2

In [ ]:
# Tests for Problem 2
assert pivot2.shape == (3, 5), f"Expected shape (3, 5), got {pivot2.shape}"
assert 'Total' in pivot2.index, "Should have a 'Total' row"
assert 'Total' in pivot2.columns, "Should have a 'Total' column"
assert abs(pivot2.loc['Total', 'Total'] - df['total_bill'].sum()) < 0.01, \
    "Grand total cell should equal the sum of all bills in the dataset"
assert pivot2.loc['Total', 'Total'] > pivot2.loc['Dinner', 'Total'], \
    "Grand total must be larger than Dinner total alone"
print("✓ All tests passed for Problem 2!")

---
## Problem 3: How Many Parties Were Served Each Day?

Operations needs staffing numbers. How many dining parties were served on each day, split by the sex of the bill payer?

Create a pivot table showing the **count of dining parties** for each **day** (rows) × **sex** (columns).

Store the result in `pivot3`.

**Hint:** Use `aggfunc='count'` with any numeric column as `values`.

**Expected shape:** (4 rows × 2 columns)

In [ ]:
# YOUR CODE HERE
pivot3 = pd.pivot_table(
    df,
    values='total_bill',
    index='day',
    columns='sex',
    aggfunc='count'
)
pivot3

In [ ]:
# Tests for Problem 3
assert pivot3.shape == (4, 2), f"Expected shape (4, 2), got {pivot3.shape}"
assert set(pivot3.columns.tolist()) == {'Male', 'Female'}, "Columns should be Male and Female"
assert pivot3.values.sum() == 244, "Total party count across all cells should be 244"
assert (pivot3.values > 0).all(), "Every day × sex combination should have at least one party"
assert pivot3.loc['Sat', 'Male'] > pivot3.loc['Fri', 'Male'], \
    "Saturday had far more parties than Friday, so Saturday Male count should be higher"
print("✓ All tests passed for Problem 3!")

---
## Problem 4: Average Tip and Bill Side by Side

You want to compare two metrics at once across sex and meal time. Create a pivot table showing **both** the mean `tip` **and** the mean `total_bill` for each **sex** (rows) × **time** (columns).

- Pass a **list** to `values` to include both metrics
- Store the result in `pivot4`

**Boilerplate code:**

```
pivot4 = pd.pivot_table(
    df, values=[___,___],
    index='___', columns='___', aggfunc='___'
)
```

**Expected shape:** (2 rows × 4 columns) — the columns will be a MultiIndex: each metric × each meal time.


In [ ]:
# YOUR CODE HERE
pivot4 = pd.pivot_table(
    df,
    values =['tip', 'total_bill'],
    index='sex',
    columns='time',
    aggfunc='mean'
)

pivot4

In [ ]:
# Tests for Problem 4
assert pivot4.shape == (2, 4), f"Expected shape (2, 4), got {pivot4.shape}"
assert isinstance(pivot4.columns, pd.MultiIndex), "Columns should be a MultiIndex"
assert 'tip' in pivot4.columns.get_level_values(0), "'tip' should appear in columns"
assert 'total_bill' in pivot4.columns.get_level_values(0), "'total_bill' should appear in columns"
assert set(pivot4.columns.get_level_values(1)) == {'Lunch', 'Dinner'}, \
    "Second level of columns should be Lunch and Dinner"
assert pivot4.loc['Male', ('tip', 'Dinner')] > 0, "Mean tip should be a positive number"
print("✓ All tests passed for Problem 4!")

---
## Problem 5: Tip Percentage by Day and Meal Period

Absolute tip amounts are misleading — a $\$5$ tip on a $\$100$ bill is stingy, but generous on a \$20 bill. The meaningful metric is **tip percentage**.

1. Add a new column `tip_pct` to `df`: `tip_pct = tip / total_bill * 100`  
`df['tip_pct'] = __`
2. Create a pivot table of **mean `tip_pct`** by **day** (rows) × **time** (columns)
3. Use `fill_value = 0` in the `pivot_table` function so that any missing day–time combinations become 0 instead of NaN
4. Store the result in `pivot5`

**Expected shape:** (4 rows × 2 columns)

> **Something to notice:** This restaurant only served *weekend* parties (Saturday and Sunday) at dinner. What does that mean for the Lunch column of those days?

In [ ]:
# YOUR CODE HERE
df['tip_pct'] = df['tip'] / df['total_bill'] * 100

pivot5= pd.pivot_table(
    df,
    values='tip_pct',
    index='day',
    columns='time',
    aggfunc='mean',
    fill_value=0
)
pivot5

In [ ]:
# Tests for Problem 5
assert 'tip_pct' in df.columns, "df should have a new 'tip_pct' column"
assert pivot5.shape == (4, 2), f"Expected shape (4, 2), got {pivot5.shape}"
assert not pivot5.isna().any().any(), "No NaN values — fill_value=0 should cover missing combos"
assert pivot5.loc['Sat', 'Lunch'] == 0, "No Saturday lunch service — should be 0 (filled)"
assert pivot5.loc['Sun', 'Lunch'] == 0, "No Sunday lunch service — should be 0 (filled)"
assert pivot5.loc['Thur', 'Lunch'] > 0, "Thursday lunch service existed — should be positive"
assert all(0 <= v <= 100 for v in pivot5.values.flatten()), \
    "All tip percentages should be between 0 and 100"
print("✓ All tests passed for Problem 5!")

---
## Problem 6: Multi-Level Index

You need a detailed view: the **mean tip** for every unique day-and-meal-period combination, split by smoking section.

Create a pivot table with:
- **Rows:** both `day` **and** `time` together — pass a list to `index`
- **Columns:** `smoker`
- **Values:** mean `tip`

Store the result in `pivot6`.

**Boilerplate code:**

```
pivot6 = pd.pivot_table(
    df, values='___',
    index=['___', '___'], columns='___', aggfunc='___'
)
```

**Think ahead:** How many rows do you expect? Remember that Saturday and Sunday had no lunch service.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

pivot6

In [ ]:
# Tests for Problem 6
assert isinstance(pivot6.index, pd.MultiIndex), "Index should be a MultiIndex (day + time)"
assert pivot6.index.names == ['day', 'time'], "Index levels should be named ['day', 'time']"
assert pivot6.shape == (6, 2), (
    f"Expected (6, 2) — 6 day-time combinations that actually exist × 2 smoker values,"
    f" got {pivot6.shape}"
)
assert ('Sat', 'Dinner') in pivot6.index, "Saturday Dinner should be present"
assert ('Sat', 'Lunch') not in pivot6.index, "Saturday Lunch should NOT exist (no data)"
assert ('Sun', 'Lunch') not in pivot6.index, "Sunday Lunch should NOT exist (no data)"
print("✓ All tests passed for Problem 6!")

---
## Problem 7: Which Day Earned the Most Total Tips?

The server wants to know which single day generated the **most tip income in total** (combining both Male and Female bill payers).

1. Create a pivot table of the **sum of tips** by **day** (rows) × **sex** (columns) — store it in `pivot_temp`
2. Define a string variable called `best_day` whose value is the best day in terms of tips. Follow this:

- Sum across sex columns (`axis=1`) to get total tips per day  
```python
#Syntax:
pivot_temp.sum(axis=1) # sums across columns to form a single row (Series) of sums
pivot_temp.sum(axis=0) # sums across rows to form a single column (Series) of sums
```
- After summing you can chain `.idxmax()` to find the winning day

**Hint:** Your intermediate pivot should have shape (4, 2).

**Examples:** 

- [Aggreating along axis](https://www.statology.org/axis-0-axis-1-python-pandas/)
- [Axis in Pandas](https://medium.com/@heyamit10/understanding-axis-in-pandas-with-examples-df5997702596)

**Boilerplate code:**
```
pivot_temp = ___
best_day = pivot_temp.sum(axis=___).idxmax()
```

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f"pivot_temp:")
display(pivot_temp)
print(f"Total tips by day:")
print(f"\nBest day: {best_day}")

In [ ]:
# Tests for Problem 7
assert pivot_temp.shape == (4, 2), f"Intermediate pivot should be (4, 2), got {pivot_temp.shape}"
assert isinstance(best_day, str), "best_day should be a string (a day name)"
assert best_day in ['Thur', 'Fri', 'Sat', 'Sun'], "best_day must be one of the four days"
assert best_day == 'Sat', (
    "Saturday had 87 dining parties — the most of any day — so it should have the highest total tips"
)
print("✓ All tests passed for Problem 7!")

---
## Problem 8: Who Tips More? A Day-by-Day Answer

**The manager's question:** *"On each day of the week, does the Male or Female bill payer leave a higher average tip percentage? Give me one answer per day."*

1. Build a pivot table of **mean `tip_pct`** (the column you created in Problem 5) by **day** (rows) × **sex** (columns) — store it in `pivot_pct`
2. For each day (each row), find which sex has the higher average tip percentage using `.idxmax(axis=1)`
3. Store the resulting Series in `top_tipper`

`top_tipper` should be a **Series** where the **index is the day** and each **value is `'Male'` or `'Female'`**.

**Hint:** `.idxmax(axis=1)` returns, for each row, the *column label* of the maximum value.

**Examples:** 

- [Aggreating along axis](https://www.statology.org/axis-0-axis-1-python-pandas/)
- [Axis in Pandas](https://medium.com/@heyamit10/understanding-axis-in-pandas-with-examples-df5997702596)

**Boilerplate code:**
```
pivot_pct = ___
top_tipper = pivot_pct.___
```

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print("Average tip percentage by day and sex:")
display(pivot_pct)
print("\nOn each day, the higher-tipping sex:")
print(top_tipper)

In [ ]:
# Tests for Problem 8
assert isinstance(top_tipper, pd.Series), "top_tipper should be a Series"
assert len(top_tipper) == 4, "Should have one entry for each of the 4 days"
assert top_tipper.index.name == 'day', "Index should be named 'day'"
assert set(top_tipper.index.tolist()) == {'Thur', 'Fri', 'Sat', 'Sun'}, \
    "Index should contain all four days"
assert all(v in ('Male', 'Female') for v in top_tipper.values), \
    "Every value must be either 'Male' or 'Female'"
assert isinstance(pivot_pct, pd.DataFrame) and pivot_pct.shape == (4, 2), \
    "pivot_pct should be a (4, 2) DataFrame"
print("✓ All tests passed for Problem 8!")